# Session 01: train a model, save it, serve it
Run this notebook from `intro/`, the copy of `sessions/01_intro/` made in the session brief, with its Python 3.12 environment active. Run the cells from top to bottom. The exercise is a regression; HighRev stays the course's classification case study.

In [1]:
from pathlib import Path
import json
import platform
import importlib.metadata as metadata
import joblib
import pandas as pd
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, r2_score

ROOT = Path.cwd()
if not (ROOT / "Dockerfile").exists():
    raise RuntimeError("Start Jupyter from intro/, the directory that holds the Dockerfile.")
print("Python:", platform.python_version())
for package in ["pandas", "scikit-learn", "joblib"]:
    print(package, metadata.version(package))


Python: 3.12.15
pandas 3.0.6
scikit-learn 1.9.1
joblib 1.6.0


## Load data and identify the target
This small dataset comes with scikit-learn; no separate download or account is required. Use raw feature values (`scaled=False`) so preprocessing is fitted on training rows. The target is a quantitative measure of disease progression after one year. This is an educational example, not a clinical service.


In [2]:
dataset = load_diabetes(as_frame=True, scaled=False)
X = dataset.data
Y = dataset.target
print("Rows:", len(X), "Features:", list(X.columns))
display(X.head())
print(dataset.DESCR)


Rows: 442 Features: ['age', 'sex', 'bmi', 'bp', 's1', 's2', 's3', 's4', 's5', 's6']


,age,sex,bmi,bp,s1,s2,s3,s4,s5,s6
0,59.0,2.0,32.1,101.0,157.0,93.2,38.0,4.0,4.8598,87.0
1,48.0,1.0,21.6,87.0,183.0,103.2,70.0,3.0,3.8918,69.0
2,72.0,2.0,30.5,93.0,156.0,93.6,41.0,4.0,4.6728,85.0
3,24.0,1.0,25.3,84.0,198.0,131.4,40.0,5.0,4.8903,89.0
4,50.0,1.0,23.0,101.0,192.0,125.4,52.0,4.0,4.2905,80.0


.. _diabetes_dataset:

Diabetes dataset
----------------

Ten baseline variables, age, sex, body mass index, average blood
pressure, and six blood serum measurements were obtained for each of n =
442 diabetes patients, as well as the response of interest, a
quantitative measure of disease progression one year after baseline.

**Data Set Characteristics:**

:Number of Instances: 442

:Number of Attributes: First 10 columns are numeric predictive values

:Target: Column 11 is a quantitative measure of disease progression one year after baseline

:Attribute Information:
    - age     age in years
    - sex
    - bmi     body mass index
    - bp      average blood pressure
    - s1      tc, total serum cholesterol
    - s2      ldl, low-density lipoproteins
    - s3      hdl, high-density lipoproteins
    - s4      tch, total cholesterol / HDL
    - s5      ltg, possibly log of serum triglycerides level
    - s6      glu, blood sugar level

Note: Each of these 10 feature variables have bee

## Fit and evaluate a simple regression
Keep 20% of rows aside for evaluation. A split is part of model evaluation, not an automated software test. Compare Ridge with predicting the training mean. Lower MAE is better; R² compares performance with a constant prediction. No metric threshold is required for this introduction.


In [3]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X, Y, test_size=0.2, random_state=42
)
model = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), Ridge(alpha=1.0))
model.fit(X_train, y_train)
mean_model = DummyRegressor(strategy="mean").fit(X_train, y_train)
for name, fitted in [("Training mean", mean_model), ("Ridge", model)]:
    predicted = fitted.predict(X_valid)
    print(name, {"MAE": mean_absolute_error(y_valid, predicted),
                 "R2": r2_score(y_valid, predicted)})


Training mean {'MAE': 64.00646146990485, 'R2': -0.011962984778542296}
Ridge {'MAE': 42.81199941834889, 'R2': 0.45414652070698236}


## Save the model and one request
Save the complete preprocessing pipeline, feature names and target together. The API will load this fitted artifact; it will not train on startup. Only load artifacts you created or trust. The request contains input features, never the target.


In [4]:
artifacts = ROOT / "artifacts"
artifacts.mkdir(exist_ok=True)
bundle = {"model": model, "feature_names": list(X.columns),
          "target": "disease_progression_after_one_year"}
joblib.dump(bundle, artifacts / "model.joblib")
row = X_valid.iloc[0]
request = {"features": {name: float(row[name]) for name in X.columns}}
(ROOT / "example.json").write_text(json.dumps(request, indent=2))
local_prediction = float(model.predict(X_valid.iloc[[0]])[0])
(ROOT / "expected.json").write_text(json.dumps({"prediction": local_prediction}, indent=2))
print("Saved:", artifacts / "model.joblib")
print("Local prediction:", local_prediction)
print(json.dumps(request, indent=2))


Saved: /Users/mikoli/highrev-mlops-course/intro/artifacts/model.joblib
Local prediction: 139.8627740520838
{
  "features": {
    "age": 61.0,
    "sex": 1.0,
    "bmi": 25.8,
    "bp": 90.0,
    "s1": 280.0,
    "s2": 195.4,
    "s3": 55.0,
    "s4": 5.0,
    "s5": 4.9972,
    "s6": 90.0
  }
}


## Continue in the session brief
Record the metrics and versions, then restart the kernel and run all cells to check that the notebook does not depend on hidden state. Back in the terminal: freeze the environment, start the local API, build the Docker image, and send `example.json` to `/predict`. Compare the returned number with `expected.json`. No test suite yet; that is session 3.